In [1]:
# Project root setup
import os
import sys
from pathlib import Path

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Could not locate the project root containing src/.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [2]:
import os
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ.setdefault("PYTHONWARNINGS", "ignore::FutureWarning:sklearn.linear_model._base")

import signal
import numpy as np
import pandas as pd
from numpy import linalg as la
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import f1_score
import random
import time
from datetime import datetime
from joblib import Parallel, delayed

from src.model import MetMulDagma, MetMulColide
import src.utils as utils

from baselines.colide import colide_ev, colide_nv
from baselines.nonnegative_dagma_linear import NonnegativeDAGMA_linear

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module=r"sklearn\.linear_model\._base")

SEED = 10
N_CPUS = max(1, int(os.environ.get("N_CPUS", os.cpu_count() or 1)))
JOBLIB_VERBOSE = max(0, int(os.environ.get("JOBLIB_VERBOSE", 10)))
SELECTED_SCENARIOS = {
    scenario.strip()
    for scenario in os.environ.get("SCENARIOS", "").split(",")
    if scenario.strip()
}
# LOAD=True only displays the saved tables; LOAD=False runs the experiments
LOAD = True
SAVE_RESULTS = False
RESULTS_PATH = './results/preliminary'
os.makedirs(RESULTS_PATH, exist_ok=True)

np.random.seed(SEED)


def log_status(message):
    timestamp = datetime.now().isoformat(timespec='seconds')
    print(f'[{timestamp} pid={os.getpid()}] {message}', flush=True)


def _handle_termination(signum, frame):
    raise KeyboardInterrupt(f"Received signal {signum}; stopping experiments")


signal.signal(signal.SIGTERM, _handle_termination)
random.seed(SEED)

In [3]:
# Hyperparameters of the nonnegative CoLiDE variants come from
# experiments/notebooks/run_algorithms.ipynb. 'lamb' values are multipliers of
# sqrt(log(N)/M) (fix_lamb=False). NOMAD and the baselines keep the
# configuration of synthetic/scripts/preliminary_exp.py.
Exps = [
    ### NONNEGATIVE COLIDE - NV (one sigma per node) ###
    {'model': MetMulColide, 'args': {'stepsize': 5e-5, 'alpha_0': .01, 'rho_0': .05, 's': 1, 'lamb': .1, 'iters_in': 30000,
     'iters_out': 10, 'beta': 1.5}, 'init': {'acyclicity': 'logdet', 'primal_opt': 'fista', 'restart': True},
     'standarize': False, 'fix_lamb': False, 'leg': 'NN-CoLiDE-NV-fista'},

    {'model': MetMulColide, 'args': {'stepsize': 3e-4, 'alpha_0': .01, 'rho_0': .05, 's': 1, 'lamb': .1, 'iters_in': 30000,
     'iters_out': 10, 'beta': 2}, 'init': {'acyclicity': 'logdet', 'primal_opt': 'adam'},
     'standarize': False, 'fix_lamb': False, 'leg': 'NN-CoLiDE-NV-adam'},

    {'model': MetMulColide, 'args': {'stepsize': 3e-4, 'alpha_0': .5, 'rho_0': .5, 's': 1, 'lamb': 1, 'iters_in': 30000,
     'iters_out': 10, 'beta': 2, 'sca_adam': False}, 'init': {'acyclicity': 'logdet', 'primal_opt': 'sca'},
     'standarize': False, 'fix_lamb': False, 'leg': 'NN-CoLiDE-NV-sca'},

    {'model': MetMulColide, 'args': {'stepsize': 3e-4, 'alpha_0': .01, 'rho_0': .05, 's': 1, 'lamb': .1, 'iters_in': 30000,
     'iters_out': 10, 'beta': 2, 'sca_adam': True}, 'init': {'acyclicity': 'logdet', 'primal_opt': 'sca'},
     'standarize': False, 'fix_lamb': False, 'leg': 'NN-CoLiDE-NV-sca-adam'},

    ### NONNEGATIVE COLIDE - EV (single shared sigma) ###
    {'model': MetMulColide, 'args': {'stepsize': 5e-5, 'alpha_0': .01, 'rho_0': .05, 's': 1, 'lamb': .1, 'iters_in': 30000,
     'iters_out': 10, 'beta': 1.5}, 'init': {'acyclicity': 'logdet', 'primal_opt': 'fista', 'restart': True, 'equal_var': True},
     'standarize': False, 'fix_lamb': False, 'leg': 'NN-CoLiDE-EV-fista'},

    {'model': MetMulColide, 'args': {'stepsize': 3e-4, 'alpha_0': .5, 'rho_0': .5, 's': 1, 'lamb': 1, 'iters_in': 30000,
     'iters_out': 10, 'beta': 2, 'sca_adam': False}, 'init': {'acyclicity': 'logdet', 'primal_opt': 'sca', 'equal_var': True},
     'standarize': False, 'fix_lamb': False, 'leg': 'NN-CoLiDE-EV-sca'},

    {'model': MetMulColide, 'args': {'stepsize': 3e-4, 'alpha_0': .01, 'rho_0': .05, 's': 1, 'lamb': .1, 'iters_in': 30000,
     'iters_out': 10, 'beta': 2, 'sca_adam': True}, 'init': {'acyclicity': 'logdet', 'primal_opt': 'sca', 'equal_var': True},
     'standarize': False, 'fix_lamb': False, 'leg': 'NN-CoLiDE-EV-sca-adam'},

    ### NOMAD (previous paper) ###
    {'model': MetMulDagma, 'args': {'stepsize': 5e-3, 'alpha_0': .1, 'rho_0': .1, 's': 1, 'lamb': .2, 'iters_in': 10000, 'step_type': 'fixed',
     'iters_out': 10, 'beta': 1.5}, 'init': {'acyclicity': 'logdet', 'primal_opt': 'adam'}, 'standarize': False,
     'fix_lamb': False, 'leg': 'NOMAD-adam'},

    {'model': MetMulDagma, 'args': {'stepsize': 1e-5, 'alpha_0': .01, 'rho_0': .01, 's': 1, 'lamb': .2, 'iters_in': 5000, 'step_type': 'fixed',
     'iters_out': 50, 'beta': 1.5}, 'init': {'acyclicity': 'logdet', 'primal_opt': 'fista', 'restart': True}, 'standarize': False,
     'fix_lamb': False, 'leg': 'NOMAD-fista'},

    ### BASELINES ###
    # NonnegDAGMA
    {'model': NonnegativeDAGMA_linear, 'init': {'loss_type': 'l2'}, 'args': {'lambda1': .05, 'T': 4, 's': [1.0, .9, .8, .7],
     'warm_iter': 2e4, 'max_iter': 7e4, 'lr': .0003}, 'standarize': False, 'leg': 'NonDAGMA'},

    # Colide
    {'model': colide_ev, 'args': {'lambda1': .05, 'T': 4, 's': [1.0, .9, .8, .7], 'warm_iter': 2e4,
     'max_iter': 7e4, 'lr': .0003}, 'standarize': False, 'leg': 'CoLiDE-EV'},

    {'model': colide_nv, 'args': {'lambda1': .05, 'T': 4, 's': [1.0, .9, .8, .7], 'warm_iter': 2e4,
     'max_iter': 7e4, 'lr': .0003}, 'standarize': False, 'leg': 'CoLiDE-NV'},
]

In [4]:
def get_lamb_value(n_nodes, n_samples, times=1):
    return np.sqrt(np.log(n_nodes) / n_samples) * times


def run_parallel_exps(data_p, exps, n_dags, scenario_name, thr=.3, verb=False):
    n_jobs = max(1, min(N_CPUS, n_dags))
    log_status(
        f'RUN scenario={scenario_name} dags={n_dags} workers={n_jobs} '
        f'joblib_verbose={JOBLIB_VERBOSE}'
    )
    t_init = time.time()

    parallel = Parallel(n_jobs=n_jobs, verbose=JOBLIB_VERBOSE)
    try:
        results = parallel(
            delayed(run_exps)(g, data_p, exps, thr=thr, verb=verb)
            for g in range(n_dags)
        )
    except (KeyboardInterrupt, SystemExit):
        backend = getattr(parallel, "_backend", None)
        if backend is not None and hasattr(backend, "terminate"):
            backend.terminate()
        raise
    finally:
        backend = getattr(parallel, "_backend", None)
        if backend is not None and hasattr(backend, "terminate"):
            backend.terminate()

    elapsed_time = (time.time() - t_init)/60
    log_status(f'DONE scenario={scenario_name} elapsed_minutes={elapsed_time:.3f}')
    return results


def run_exps(g, data_p, exps, thr=.3, verb=False):
    A_true, _, X = utils.simulate_sem(**data_p)
    A_true_bin = utils.to_bin(A_true, thr)
    X_std = utils.standarize(X)
    X_scale = X.std(axis=0)
    A_true_std = A_true * X_scale[:, None] / X_scale[None, :]

    M, N = X.shape

    Z = X - X @ A_true
    fidelity = 1/data_p['n_samples']*la.norm(Z, 'fro')**2
    fidelity_std = 1/data_p['n_samples']*la.norm(X_std - X_std @ A_true_std, 'fro')**2

    Sigma_hat = la.norm(Z, axis=0) / np.sqrt(M)

    log_status(
        f'DAG {g}: fidelity={fidelity:.3f} fidelity_std={fidelity_std:.3f} '
        f'sigma_min={np.min(Sigma_hat):.4f} sigma_max={np.max(Sigma_hat):.4f}'
    )

    shd, tpr, fdr, fscore, sid_norm, err, rel_err, err_th, acyc, runtime = [
        np.zeros(len(exps)) for _ in range(10)
    ]
    for i, exp in enumerate(exps):
        standardized = exp.get('standarize', False)
        X_aux = X_std if standardized else X
        A_true_aux = A_true_std if standardized else A_true

        args = exp['args'].copy()
        if 'fix_lamb' in exp.keys() and not exp['fix_lamb']:
            if 'lamb' in args:
                args['lamb'] = get_lamb_value(N, M, args['lamb'])
            elif 'lambda1' in args:
                args['lambda1'] = get_lamb_value(N, M, args['lambda1'])

        if 'know_var' in exp.keys() and exp['know_var']:
            args['Sigma'] = data_p['var']

        model = None
        try:
            model = exp['model'](**exp['init']) if 'init' in exp.keys() else exp['model']()
            t_i = time.time()
            model.fit(X_aux, **args)
            t_solved = time.time() - t_i

            A_est = model.W_est
            if getattr(model, 'cycle_repair_applied_', False):
                log_status(
                    f'DAG {g} method={exp["leg"]} exact-DAG safeguard removed '
                    f'{model.cycle_repair_count_} edge(s): '
                    f'{model.cycle_repair_edges_}'
                )
        except Exception as exc:
            raise RuntimeError(
                f'DAG {g} method={exp["leg"]} failed: '
                f'{type(exc).__name__}: {exc}'
            ) from exc

        A_est_bin = utils.to_bin(A_est, thr)
        try:
            shd[i], tpr[i], fdr[i], sid_norm[i] = utils.count_accuracy(
                A_true_bin,
                A_est_bin,
                compute_sid=True,
                sid_normalize=True,
            )
        except ValueError:
            shd[i], tpr[i], fdr[i] = utils.count_accuracy(A_true_bin, A_est_bin)
            sid_norm[i] = np.nan
        fscore[i] = f1_score(A_true_bin.flatten(), A_est_bin.flatten())
        err[i] = utils.compute_norm_sq_err(A_true_aux, A_est)
        rel_err[i] = la.norm(A_true_aux - A_est, 'fro')**2 / la.norm(A_true_aux, 'fro')**2
        A_est_th = A_est.copy()
        A_est_th[np.abs(A_est_th) < thr] = 0
        err_th[i] = utils.compute_norm_sq_err(A_true_aux, A_est_th)
        acyc[i] = model.dagness(A_est) if model is not None and hasattr(model, 'dagness') else float(not utils.is_dag(A_est_bin))
        runtime[i] = t_solved

        if verb and (g % N_CPUS == 0):
            sid_text = f'{sid_norm[i]:.3f}' if np.isfinite(sid_norm[i]) else 'nan'
            log_status(
                f'DAG {g} method={exp["leg"]} standardized={standardized} '
                f'shd={shd[i]} tpr={tpr[i]:.3f} fdr={fdr[i]:.3f} sid={sid_text} '
                f'err={err[i]:.3f} rel_err={rel_err[i]:.3f} err_th={err_th[i]:.3f} '
                f'time={runtime[i]:.3f}'
            )

    return shd, tpr, fdr, fscore, sid_norm, err, rel_err, err_th, acyc, runtime


def preliminary_results_prefix(scenario_name):
    return f'{RESULTS_PATH}/preliminary_{scenario_name}'


def load_preliminary_results(scenario_name):
    tables = {}
    exps_leg = None
    for agg in ('mean', 'median'):
        file_name = f'{preliminary_results_prefix(scenario_name)}_{agg}.csv'
        if not os.path.exists(file_name):
            raise FileNotFoundError(f'Results file not found: {file_name}')
        tables[agg] = pd.read_csv(file_name)
        if 'leg' not in tables[agg].columns:
            raise ValueError(f'Results file has no experiment legend column: {file_name}')

        table_exps_leg = tables[agg]['leg'].astype(str).tolist()
        if exps_leg is None:
            exps_leg = table_exps_leg
        elif table_exps_leg != exps_leg:
            raise ValueError(f'Experiment legends differ between saved tables for {scenario_name}')

        print(f'Loaded {agg} results from {file_name}')
        display(tables[agg])
    return tables, exps_leg


def run_or_load_preliminary_results(data_p, exps, n_dags, scenario_name, thr=.3, verb=False):
    if SELECTED_SCENARIOS and scenario_name not in SELECTED_SCENARIOS:
        log_status(f'SKIP scenario={scenario_name} selected={sorted(SELECTED_SCENARIOS)}')
        return None, None, None

    standardized_count = sum(exp.get('standarize', False) for exp in exps)
    mode = 'load' if LOAD else 'run'
    var = np.asarray(data_p['var'])
    var_text = f'{var:.3g}' if var.ndim == 0 else f'hetero[{var.min():.3g},{var.max():.3g}] mean={var.mean():.3g}'
    log_status(
        f'START scenario={scenario_name} mode={mode} nodes={data_p["n_nodes"]} '
        f'samples={data_p["n_samples"]} edges={data_p["edges"]} var={var_text} dags={n_dags} '
        f'standardized_methods={standardized_count}/{len(exps)}'
    )

    if LOAD:
        tables, exps_leg = load_preliminary_results(scenario_name)
        log_status(f'LOADED scenario={scenario_name} methods={len(exps_leg)}')
        return None, tables, exps_leg

    results = run_parallel_exps(
        data_p,
        exps,
        n_dags,
        scenario_name=scenario_name,
        thr=thr,
        verb=verb,
    )
    shd, tpr, fdr, fscore, sid_norm, err, rel_err, err_th, acyc, runtime = zip(*results)
    metrics = {
        'shd': shd,
        'tpr': tpr,
        'fdr': fdr,
        'fscore': fscore,
        'sid_norm': sid_norm,
        'err': err,
        'rel_err': rel_err,
        'err_th': err_th,
        'acyc': acyc,
        'time': runtime,
    }

    exps_leg = [exp['leg'] for exp in exps]
    file_prefix = preliminary_results_prefix(scenario_name) if SAVE_RESULTS else None
    utils.display_results(exps_leg, metrics, agg='mean', file_name=f'{file_prefix}_mean' if file_prefix else None)
    utils.display_results(exps_leg, metrics, agg='median', file_name=f'{file_prefix}_median' if file_prefix else None)
    log_status(f'SAVED scenario={scenario_name} prefix={file_prefix}')
    return metrics, None, exps_leg

## CASE 1 - N=100, Homocedastic, var=1, Weights - [0.5, 1]

In [5]:
N = 100
SCENARIO_NAME = 'er4_N100_var1'

n_dags = 100
verb = True
data_params = {
    'n_nodes': N,
    'n_samples': 1000,
    'graph_type': 'er',
    'edges': 4*N,
    'edge_type': 'positive',
    'w_range': (.5, 1),
    'var': 1
}
metrics, tables, exps_leg = run_or_load_preliminary_results(data_params, Exps, n_dags, SCENARIO_NAME, thr=.3, verb=verb)

[2026-09-11T16:40:21 pid=2901929] START scenario=er4_N100_var1 mode=load nodes=100 samples=1000 edges=400 var=1 dags=100 standardized_methods=0/12
Loaded mean results from ./results/nonneg_colide/preliminary/preliminary_er4_N100_var1_mean.csv


,leg,shd,tpr,fdr,fscore,sid_norm,err,rel_err,err_th,acyc,time
0,NN-CoLiDE-NV-fista,1.1000 ± 4.3509,0.9978 ± 0.0087,0.0011 ± 0.0052,0.9984 ± 0.0063,0.2766 ± 1.1207,0.0075 ± 0.0091,0.0081 ± 0.0090,0.0048 ± 0.0104,0.0109 ± 0.0166,279.3401 ± 92.8552
1,NN-CoLiDE-NV-adam,47.1500 ± 28.0358,0.8917 ± 0.0556,0.0113 ± 0.0160,0.9371 ± 0.0371,1.1023 ± 1.0097,0.1088 ± 0.0616,0.1091 ± 0.0567,0.1234 ± 0.0672,0.0027 ± 0.0006,562.9113 ± 84.3147
2,NN-CoLiDE-NV-sca,137.5600 ± 47.4915,0.7056 ± 0.1019,0.1178 ± 0.0943,0.7828 ± 0.0968,20.7984 ± 2.8315,0.3793 ± 0.1841,0.3390 ± 0.0966,0.4018 ± 0.1917,0.0099 ± 0.0111,285.9672 ± 111.2355
3,NN-CoLiDE-NV-sca-adam,0.8100 ± 1.1198,0.9985 ± 0.0017,0.0020 ± 0.0028,0.9983 ± 0.0022,0.6877 ± 0.7825,0.0067 ± 0.0024,0.0072 ± 0.0024,0.0037 ± 0.0024,0.0024 ± 0.0006,35.3131 ± 8.5305
4,NN-CoLiDE-EV-fista,0.0600 ± 0.4200,0.9999 ± 0.0006,0.0001 ± 0.0011,0.9999 ± 0.0008,0.0451 ± 0.2609,0.0051 ± 0.0011,0.0056 ± 0.0011,0.0021 ± 0.0010,0.0065 ± 0.0023,95.6559 ± 21.6596
5,NN-CoLiDE-EV-sca,52.0400 ± 45.2068,0.8854 ± 0.0857,0.0276 ± 0.0410,0.9258 ± 0.0658,9.2183 ± 4.5818,0.1182 ± 0.1142,0.1311 ± 0.0965,0.1426 ± 0.1312,0.0098 ± 0.0165,159.2420 ± 68.3740
6,NN-CoLiDE-EV-sca-adam,0.1400 ± 0.5834,0.9998 ± 0.0008,0.0004 ± 0.0015,0.9997 ± 0.0011,0.1193 ± 0.3917,0.0052 ± 0.0009,0.0057 ± 0.0009,0.0022 ± 0.0009,0.0026 ± 0.0005,32.7726 ± 12.0326
7,NOMAD-adam,0.0300 ± 0.1706,0.9999 ± 0.0004,0.0000 ± 0.0000,1.0000 ± 0.0002,0.0026 ± 0.0249,0.0038 ± 0.0004,0.0051 ± 0.0005,0.0024 ± 0.0004,0.0081 ± 0.0019,28.2479 ± 9.7308
8,NOMAD-fista,0.2000 ± 0.5099,0.9995 ± 0.0012,0.0004 ± 0.0012,0.9996 ± 0.0012,0.1870 ± 0.5011,0.0044 ± 0.0017,0.0058 ± 0.0018,0.0030 ± 0.0017,0.0000 ± 0.0000,29.4597 ± 12.3647
9,NonDAGMA,0.7000 ± 1.0724,0.9983 ± 0.0025,0.0007 ± 0.0017,0.9988 ± 0.0019,0.4636 ± 0.7938,0.0072 ± 0.0032,0.0120 ± 0.0034,0.0065 ± 0.0032,0.0001 ± 0.0000,28.5533 ± 11.6079


Loaded median results from ./results/nonneg_colide/preliminary/preliminary_er4_N100_var1_median.csv


,leg,shd,tpr,fdr,fscore,sid_norm,err,rel_err,err_th,acyc,time
0,NN-CoLiDE-NV-fista,0.0000 ± 4.3509,1.0000 ± 0.0087,0.0000 ± 0.0052,1.0000 ± 0.0063,0.0000 ± 1.1207,0.0050 ± 0.0091,0.0056 ± 0.0090,0.0020 ± 0.0104,0.0072 ± 0.0166,263.2029 ± 92.8552
1,NN-CoLiDE-NV-adam,43.5000 ± 28.0358,0.8957 ± 0.0556,0.0053 ± 0.0160,0.9408 ± 0.0371,0.7800 ± 1.0097,0.0972 ± 0.0616,0.0984 ± 0.0567,0.1115 ± 0.0672,0.0027 ± 0.0006,576.6815 ± 84.3147
2,NN-CoLiDE-NV-sca,130.0000 ± 47.4915,0.7247 ± 0.1019,0.1023 ± 0.0943,0.8024 ± 0.0968,20.7850 ± 2.8315,0.3494 ± 0.1841,0.3231 ± 0.0966,0.3681 ± 0.1917,0.0066 ± 0.0111,321.3213 ± 111.2355
3,NN-CoLiDE-NV-sca-adam,0.0000 ± 1.1198,1.0000 ± 0.0017,0.0000 ± 0.0028,1.0000 ± 0.0022,0.0000 ± 0.7825,0.0054 ± 0.0024,0.0060 ± 0.0024,0.0023 ± 0.0024,0.0024 ± 0.0006,32.5556 ± 8.5305
4,NN-CoLiDE-EV-fista,0.0000 ± 0.4200,1.0000 ± 0.0006,0.0000 ± 0.0011,1.0000 ± 0.0008,0.0000 ± 0.2609,0.0049 ± 0.0011,0.0055 ± 0.0011,0.0020 ± 0.0010,0.0064 ± 0.0023,90.6616 ± 21.6596
5,NN-CoLiDE-EV-sca,38.0000 ± 45.2068,0.9095 ± 0.0857,0.0160 ± 0.0410,0.9466 ± 0.0658,8.5450 ± 4.5818,0.0876 ± 0.1142,0.1044 ± 0.0965,0.1062 ± 0.1312,0.0053 ± 0.0165,145.7563 ± 68.3740
6,NN-CoLiDE-EV-sca-adam,0.0000 ± 0.5834,1.0000 ± 0.0008,0.0000 ± 0.0015,1.0000 ± 0.0011,0.0000 ± 0.3917,0.0050 ± 0.0009,0.0055 ± 0.0009,0.0020 ± 0.0009,0.0026 ± 0.0005,30.3558 ± 12.0326
7,NOMAD-adam,0.0000 ± 0.1706,1.0000 ± 0.0004,0.0000 ± 0.0000,1.0000 ± 0.0002,0.0000 ± 0.0249,0.0037 ± 0.0004,0.0051 ± 0.0005,0.0023 ± 0.0004,0.0079 ± 0.0019,27.5677 ± 9.7308
8,NOMAD-fista,0.0000 ± 0.5099,1.0000 ± 0.0012,0.0000 ± 0.0012,1.0000 ± 0.0012,0.0000 ± 0.5011,0.0041 ± 0.0017,0.0055 ± 0.0018,0.0025 ± 0.0017,0.0000 ± 0.0000,24.0460 ± 12.3647
9,NonDAGMA,0.0000 ± 1.0724,1.0000 ± 0.0025,0.0000 ± 0.0017,1.0000 ± 0.0019,0.0000 ± 0.7938,0.0061 ± 0.0032,0.0111 ± 0.0034,0.0053 ± 0.0032,0.0001 ± 0.0000,22.8308 ± 11.6079


[2026-09-11T16:40:21 pid=2901929] LOADED scenario=er4_N100_var1 methods=12


## CASE 2 - N=100, Homocedastic, var=15, Weights - [0.5, 1]

In [6]:
N = 100
SCENARIO_NAME = 'er4_N100_var15'

n_dags = 100
verb = True
data_params = {
    'n_nodes': N,
    'n_samples': 1000,
    'graph_type': 'er',
    'edges': 4*N,
    'edge_type': 'positive',
    'w_range': (.5, 1),
    'var': 15
}
metrics, tables, exps_leg = run_or_load_preliminary_results(data_params, Exps, n_dags, SCENARIO_NAME, thr=.3, verb=verb)

[2026-09-11T17:27:06 pid=2901929] START scenario=er4_N100_var15 mode=load nodes=100 samples=1000 edges=400 var=15 dags=100 standardized_methods=0/12
Loaded mean results from ./results/nonneg_colide/preliminary/preliminary_er4_N100_var15_mean.csv


,leg,shd,tpr,fdr,fscore,sid_norm,err,rel_err,err_th,acyc,time
0,NN-CoLiDE-NV-fista,14.8400 ± 38.8257,0.9763 ± 0.0622,0.0135 ± 0.0393,0.9808 ± 0.0498,0.6776 ± 2.7202,0.0394 ± 0.0779,0.0404 ± 0.0778,0.0377 ± 0.0911,0.1246 ± 0.5719,198.9644 ± 81.7426
1,NN-CoLiDE-NV-adam,10.7000 ± 11.4013,0.9753 ± 0.0256,0.0024 ± 0.0057,0.9862 ± 0.0149,0.3804 ± 0.7271,0.0281 ± 0.0241,0.0307 ± 0.0242,0.0312 ± 0.0284,0.0038 ± 0.0015,282.2170 ± 66.0762
2,NN-CoLiDE-NV-sca,46.8900 ± 54.0061,0.9159 ± 0.1080,0.0583 ± 0.0485,0.9266 ± 0.0857,10.5037 ± 5.4541,0.1234 ± 0.1251,0.1216 ± 0.1118,0.1289 ± 0.1554,0.0051 ± 0.0070,162.4833 ± 67.8973
3,NN-CoLiDE-NV-sca-adam,0.0500 ± 0.2598,0.9999 ± 0.0005,0.0001 ± 0.0006,0.9999 ± 0.0006,0.0517 ± 0.2547,0.0089 ± 0.0009,0.0091 ± 0.0008,0.0023 ± 0.0005,0.0035 ± 0.0010,31.6295 ± 9.8945
4,NN-CoLiDE-EV-fista,0.0000 ± 0.0000,1.0000 ± 0.0000,0.0000 ± 0.0000,1.0000 ± 0.0000,0.0000 ± 0.0000,0.0089 ± 0.0009,0.0091 ± 0.0009,0.0022 ± 0.0002,0.0135 ± 0.0192,87.6298 ± 30.5099
5,NN-CoLiDE-EV-sca,19.6300 ± 41.1402,0.9620 ± 0.0876,0.0185 ± 0.0242,0.9699 ± 0.0659,4.2435 ± 5.1065,0.0563 ± 0.0877,0.0628 ± 0.0846,0.0557 ± 0.1124,0.0043 ± 0.0098,129.4287 ± 63.2337
6,NN-CoLiDE-EV-sca-adam,0.0100 ± 0.0995,1.0000 ± 0.0003,0.0000 ± 0.0003,1.0000 ± 0.0003,0.0109 ± 0.1085,0.0089 ± 0.0009,0.0091 ± 0.0008,0.0022 ± 0.0003,0.0034 ± 0.0011,34.7256 ± 10.8526
7,NOMAD-adam,0.0700 ± 0.4529,0.9999 ± 0.0004,0.0002 ± 0.0011,0.9999 ± 0.0007,0.0388 ± 0.2224,0.0103 ± 0.0013,0.0104 ± 0.0012,0.0024 ± 0.0007,0.0401 ± 0.0092,32.5258 ± 10.7258
8,NOMAD-fista,0.2000 ± 0.9274,0.9997 ± 0.0011,0.0004 ± 0.0019,0.9997 ± 0.0015,0.1054 ± 0.4199,0.0118 ± 0.0037,0.0118 ± 0.0038,0.0029 ± 0.0019,0.0000 ± 0.0001,41.2239 ± 13.2098
9,NonDAGMA,7.7900 ± 25.0257,0.9911 ± 0.0219,0.0106 ± 0.0360,0.9901 ± 0.0288,1.1884 ± 1.7594,0.0273 ± 0.0469,0.0282 ± 0.0443,0.0196 ± 0.0444,0.1101 ± 0.3668,22.8577 ± 10.0739


Loaded median results from ./results/nonneg_colide/preliminary/preliminary_er4_N100_var15_median.csv


,leg,shd,tpr,fdr,fscore,sid_norm,err,rel_err,err_th,acyc,time
0,NN-CoLiDE-NV-fista,1.0000 ± 38.8257,0.9975 ± 0.0622,0.0000 ± 0.0393,0.9982 ± 0.0498,0.0000 ± 2.7202,0.0117 ± 0.0779,0.0120 ± 0.0778,0.0052 ± 0.0911,0.0199 ± 0.5719,205.9161 ± 81.7426
1,NN-CoLiDE-NV-adam,8.0000 ± 11.4013,0.9822 ± 0.0256,0.0000 ± 0.0057,0.9898 ± 0.0149,0.0450 ± 0.7271,0.0233 ± 0.0241,0.0258 ± 0.0242,0.0263 ± 0.0284,0.0040 ± 0.0015,303.1462 ± 66.0762
2,NN-CoLiDE-NV-sca,26.0000 ± 54.0061,0.9533 ± 0.1080,0.0475 ± 0.0485,0.9570 ± 0.0857,9.6950 ± 5.4541,0.0759 ± 0.1251,0.0795 ± 0.1118,0.0713 ± 0.1554,0.0031 ± 0.0070,156.0988 ± 67.8973
3,NN-CoLiDE-NV-sca-adam,0.0000 ± 0.2598,1.0000 ± 0.0005,0.0000 ± 0.0006,1.0000 ± 0.0006,0.0000 ± 0.2547,0.0089 ± 0.0009,0.0090 ± 0.0008,0.0022 ± 0.0005,0.0035 ± 0.0010,30.9422 ± 9.8945
4,NN-CoLiDE-EV-fista,0.0000 ± 0.0000,1.0000 ± 0.0000,0.0000 ± 0.0000,1.0000 ± 0.0000,0.0000 ± 0.0000,0.0088 ± 0.0009,0.0090 ± 0.0009,0.0022 ± 0.0002,0.0105 ± 0.0192,80.9551 ± 30.5099
5,NN-CoLiDE-EV-sca,7.0000 ± 41.1402,0.9878 ± 0.0876,0.0091 ± 0.0242,0.9894 ± 0.0659,2.7700 ± 5.1065,0.0302 ± 0.0877,0.0386 ± 0.0846,0.0251 ± 0.1124,0.0020 ± 0.0098,128.2721 ± 63.2337
6,NN-CoLiDE-EV-sca-adam,0.0000 ± 0.0995,1.0000 ± 0.0003,0.0000 ± 0.0003,1.0000 ± 0.0003,0.0000 ± 0.1085,0.0088 ± 0.0009,0.0090 ± 0.0008,0.0022 ± 0.0003,0.0033 ± 0.0011,32.1337 ± 10.8526
7,NOMAD-adam,0.0000 ± 0.4529,1.0000 ± 0.0004,0.0000 ± 0.0011,1.0000 ± 0.0007,0.0000 ± 0.2224,0.0101 ± 0.0013,0.0102 ± 0.0012,0.0023 ± 0.0007,0.0398 ± 0.0092,31.3443 ± 10.7258
8,NOMAD-fista,0.0000 ± 0.9274,1.0000 ± 0.0011,0.0000 ± 0.0019,1.0000 ± 0.0015,0.0000 ± 0.4199,0.0114 ± 0.0037,0.0114 ± 0.0038,0.0025 ± 0.0019,0.0000 ± 0.0001,37.6356 ± 13.2098
9,NonDAGMA,1.0000 ± 25.0257,0.9975 ± 0.0219,0.0000 ± 0.0360,0.9976 ± 0.0288,0.9100 ± 1.7594,0.0136 ± 0.0469,0.0148 ± 0.0443,0.0070 ± 0.0444,0.0030 ± 0.3668,21.0412 ± 10.0739


[2026-09-11T17:27:06 pid=2901929] LOADED scenario=er4_N100_var15 methods=12


## CASE 3 - N=100, Heterocedastic, mean var=1, Weights - [0.5, 1]

In [7]:
N = 100
SCENARIO_NAME = 'er4_N100_hetero_var1'

# Per-node variances drawn once (seeded) and shared by all DAGs, as in the original script.
var = np.random.uniform(low=0.5, high=1.5, size=N)

n_dags = 100
verb = True
data_params = {
    'n_nodes': N,
    'n_samples': 1000,
    'graph_type': 'er',
    'edges': 4*N,
    'edge_type': 'positive',
    'w_range': (.5, 1),
    'var': var
}
metrics, tables, exps_leg = run_or_load_preliminary_results(data_params, Exps, n_dags, SCENARIO_NAME, thr=.3, verb=verb)

[2026-09-11T17:35:34 pid=2901929] START scenario=er4_N100_hetero_var1 mode=load nodes=100 samples=1000 edges=400 var=hetero[0.504,1.49] mean=0.985 dags=100 standardized_methods=0/12
Loaded mean results from ./results/nonneg_colide/preliminary/preliminary_er4_N100_hetero_var1_mean.csv


,leg,shd,tpr,fdr,fscore,sid_norm,err,rel_err,err_th,acyc,time
0,NN-CoLiDE-NV-fista,3.7600 ± 6.7840,0.9937 ± 0.0122,0.0059 ± 0.0072,0.9939 ± 0.0088,1.4113 ± 1.4999,0.0138 ± 0.0142,0.0142 ± 0.0138,0.0110 ± 0.0146,0.0106 ± 0.0179,134.1441 ± 38.5975
1,NN-CoLiDE-NV-adam,49.3700 ± 26.2243,0.8873 ± 0.0536,0.0133 ± 0.0120,0.9337 ± 0.0341,1.6805 ± 1.2515,0.1133 ± 0.0577,0.1132 ± 0.0537,0.1284 ± 0.0632,0.0025 ± 0.0006,298.7670 ± 38.6034
2,NN-CoLiDE-NV-sca,140.7100 ± 48.4449,0.7026 ± 0.0937,0.1131 ± 0.0400,0.7817 ± 0.0760,20.7270 ± 2.9397,0.3807 ± 0.1202,0.3445 ± 0.0923,0.4077 ± 0.1405,0.0131 ± 0.0174,195.0726 ± 69.1429
3,NN-CoLiDE-NV-sca-adam,2.3200 ± 2.6225,0.9965 ± 0.0035,0.0051 ± 0.0055,0.9957 ± 0.0043,1.2315 ± 1.1607,0.0104 ± 0.0056,0.0109 ± 0.0055,0.0076 ± 0.0058,0.0023 ± 0.0005,25.0684 ± 6.3499
4,NN-CoLiDE-EV-fista,2.6500 ± 2.9542,0.9964 ± 0.0036,0.0059 ± 0.0065,0.9952 ± 0.0047,1.2634 ± 1.2186,0.0114 ± 0.0064,0.0118 ± 0.0062,0.0083 ± 0.0063,0.0058 ± 0.0041,73.7741 ± 21.9866
5,NN-CoLiDE-EV-sca,54.8900 ± 39.4499,0.8785 ± 0.0772,0.0273 ± 0.0237,0.9220 ± 0.0562,9.9423 ± 4.3167,0.1201 ± 0.0785,0.1337 ± 0.0722,0.1478 ± 0.1005,0.0070 ± 0.0073,120.8148 ± 50.3879
6,NN-CoLiDE-EV-sca-adam,2.3200 ± 2.6641,0.9965 ± 0.0036,0.0051 ± 0.0054,0.9957 ± 0.0043,1.2504 ± 1.2132,0.0106 ± 0.0056,0.0110 ± 0.0054,0.0077 ± 0.0059,0.0018 ± 0.0006,29.0597 ± 8.9608
7,NOMAD-adam,2.1700 ± 2.5340,0.9965 ± 0.0037,0.0047 ± 0.0053,0.9959 ± 0.0043,1.2447 ± 1.2672,0.0094 ± 0.0059,0.0106 ± 0.0057,0.0081 ± 0.0061,0.0050 ± 0.0017,22.6587 ± 5.7814
8,NOMAD-fista,3.9000 ± 8.0430,0.9950 ± 0.0116,0.0076 ± 0.0094,0.9937 ± 0.0102,1.1926 ± 1.1493,0.0147 ± 0.0212,0.0159 ± 0.0214,0.0121 ± 0.0164,0.0000 ± 0.0000,25.3017 ± 7.9550
9,NonDAGMA,2.8700 ± 2.4152,0.9945 ± 0.0041,0.0043 ± 0.0046,0.9951 ± 0.0039,1.3748 ± 1.0632,0.0120 ± 0.0055,0.0170 ± 0.0053,0.0119 ± 0.0058,0.0001 ± 0.0004,26.4323 ± 7.6452


Loaded median results from ./results/nonneg_colide/preliminary/preliminary_er4_N100_hetero_var1_median.csv


,leg,shd,tpr,fdr,fscore,sid_norm,err,rel_err,err_th,acyc,time
0,NN-CoLiDE-NV-fista,1.5000 ± 6.7840,0.9974 ± 0.0122,0.0026 ± 0.0072,0.9968 ± 0.0088,1.1950 ± 1.4999,0.0091 ± 0.0142,0.0096 ± 0.0138,0.0063 ± 0.0146,0.0049 ± 0.0179,133.0906 ± 38.5975
1,NN-CoLiDE-NV-adam,43.0000 ± 26.2243,0.8986 ± 0.0536,0.0089 ± 0.0120,0.9421 ± 0.0341,1.5750 ± 1.2515,0.1032 ± 0.0577,0.1052 ± 0.0537,0.1145 ± 0.0632,0.0024 ± 0.0006,299.0757 ± 38.6034
2,NN-CoLiDE-NV-sca,128.0000 ± 48.4449,0.7161 ± 0.0937,0.1037 ± 0.0400,0.7992 ± 0.0760,20.9450 ± 2.9397,0.3626 ± 0.1202,0.3323 ± 0.0923,0.3799 ± 0.1405,0.0069 ± 0.0174,190.0904 ± 69.1429
3,NN-CoLiDE-NV-sca-adam,2.0000 ± 2.6225,0.9975 ± 0.0035,0.0026 ± 0.0055,0.9964 ± 0.0043,1.2150 ± 1.1607,0.0092 ± 0.0056,0.0096 ± 0.0055,0.0061 ± 0.0058,0.0022 ± 0.0005,25.6275 ± 6.3499
4,NN-CoLiDE-EV-fista,1.5000 ± 2.9542,0.9974 ± 0.0036,0.0038 ± 0.0065,0.9968 ± 0.0047,1.1950 ± 1.2186,0.0096 ± 0.0064,0.0101 ± 0.0062,0.0066 ± 0.0063,0.0046 ± 0.0041,69.9820 ± 21.9866
5,NN-CoLiDE-EV-sca,46.0000 ± 39.4499,0.8986 ± 0.0772,0.0212 ± 0.0237,0.9347 ± 0.0562,9.3800 ± 4.3167,0.1025 ± 0.0785,0.1137 ± 0.0722,0.1229 ± 0.1005,0.0053 ± 0.0073,110.2014 ± 50.3879
6,NN-CoLiDE-EV-sca-adam,1.5000 ± 2.6641,0.9975 ± 0.0036,0.0027 ± 0.0054,0.9969 ± 0.0043,1.2400 ± 1.2132,0.0093 ± 0.0056,0.0097 ± 0.0054,0.0060 ± 0.0059,0.0018 ± 0.0006,28.4527 ± 8.9608
7,NOMAD-adam,1.0000 ± 2.5340,0.9975 ± 0.0037,0.0025 ± 0.0053,0.9974 ± 0.0043,1.1900 ± 1.2672,0.0076 ± 0.0059,0.0089 ± 0.0057,0.0061 ± 0.0061,0.0047 ± 0.0017,20.0903 ± 5.7814
8,NOMAD-fista,3.0000 ± 8.0430,0.9973 ± 0.0116,0.0062 ± 0.0094,0.9952 ± 0.0102,1.2400 ± 1.1493,0.0119 ± 0.0212,0.0129 ± 0.0214,0.0096 ± 0.0164,0.0000 ± 0.0000,21.8592 ± 7.9550
9,NonDAGMA,2.0000 ± 2.4152,0.9950 ± 0.0041,0.0026 ± 0.0046,0.9962 ± 0.0039,1.3050 ± 1.0632,0.0105 ± 0.0055,0.0156 ± 0.0053,0.0104 ± 0.0058,0.0001 ± 0.0004,22.9316 ± 7.6452


[2026-09-11T17:35:34 pid=2901929] LOADED scenario=er4_N100_hetero_var1 methods=12
